In [ ]:
import os
import openai
import json
import random
import time
import re
import pickle
from transformers import LlamaTokenizer
from tqdm import tqdm
import pandas as pd
import warnings
from openai import OpenAI
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

load_dotenv()

In [ ]:
def get_completion(message,model):
    
    client = OpenAI()


    response = client.chat.completions.create(
        model=model,
        messages=message
    )

    # Using regex to find a 0 or 1 in the response
    match = re.search(r'\b(0|1)\b', response.choices[0].message.content)
    
    if match:
        return int(match.group(1))  # Returns the matched 0 or 1 as an integer
    else:
        print("Valid response not found, please check the model's output or adjust the prompt.")
        return None 

def extract_answer(text):
    # Split the text using '### Answer:' as the delimiter
    parts = text.split('Answer:')
    
    # Return the part after the '### Answer:' marker, stripped of any leading or trailing whitespace
    return parts[1].strip() if len(parts) > 1 else ''

prompt = """You are a helpful and precise assistant for comparing the quality of the Assistant 1's answer and the Assistant 2's answer.
[The start of Question] 
{question}
[The end of Question]

[The Start of Assistant 1's Answer]
{assistant_answer1}
[The End of Assistant 1's Answer]

[The Start of Assistant 2's Answer]
{assistant_answer2}
[The End of Assistant 2's Answer]

We would like to request your feedback on the performance of the AI assistant in response to the user question.
"""

In [ ]:
import itertools
import numpy as np
subs = ['anatomy', 'clinical_knowledge','college_biology', 'college_medicine','medical_genetics']

for sub in subs:

    with open(f"data/{sub}_alpaca_num=100.json", 'r') as f:
        alpaca = json.load(f)
        
    with open(f"data/{sub}_llama1_num=100.json", 'r') as f:
            llama1 = json.load(f)
    
    with open(f"data/{sub}_llama2_num=100.json", 'r') as f:
            llama2 = json.load(f)
    
    with open(f"data/{sub}_gpt4o_num=100.json", 'r') as f:
            gpt4o = json.load(f)
    
    with open(f"data/{sub}_gpt3_num=100.json", 'r') as f:
           gpt3 = json.load(f)
    
    print(sub)    
    vnames = ['gpt3','alpaca', 'gpt4o', 'llama1',  'llama2']
    num = 100
    model = "gpt-4"
    out_data = []
    
    with open(f"eval/{sub}_num=100.json", 'r') as f:
        pairs = json.load(f)
    # Iterate over all unique pairs of variable names
    for variable_name1, variable_name2 in itertools.combinations(vnames, 2):
        if np.random.binomial(1, p=1.0):
            found = any(pair for pair in pairs if pair.get("variable_name1") == variable_name1 and pair.get("variable_name2") == variable_name2)
            if found:
                 continue
            data1 = globals()[variable_name1]
            data2 = globals()[variable_name2]
            print(f"\n============== Pair: {variable_name1} vs {variable_name2} ===============\n")
            
            out0 = np.zeros(num)
            for i in tqdm(range(num)):
                question = data1[i]['prompt']
    
                prompt_str = (prompt.format(
                    question=question,
                    assistant_answer1=data1[i]['response'],
                    assistant_answer2=data2[i]['response'],
                    ) + 
                    "Please first output a single line containing one value: 1 if the assistant 1's answer is better than the assistant 2's answer; 0 if not). The only output i want is a single value. " )
    
                message = [{"role": "system", "content": "You are a healthcare professional."},
                           {"role": "user", "content": prompt_str}]
                gpt4 = get_completion(message, model)
                out0[i] = int(gpt4)
    
            
            print(out0)
            
            out_data.append({
                'variable_name1': variable_name1,
                'variable_name2': variable_name2,
                'GPT4_output': out0
            })
    
    file_path = f"eval/{sub}_num={num}.json"
    print(file_path)
    for entry in out_data:
        if isinstance(entry['GPT4_output'], np.ndarray):
            entry['GPT4_output'] = entry['GPT4_output'].tolist()
    
    # Saving the data to a JSON file
    with open(file_path, 'a') as json_file:
        json.dump(out_data, json_file, indent=4)